# 01 — Transformación de Bronze a Silver

Este notebook limpia y valida los datos de ventas y clientes de la capa **Bronze**, y persiste el resultado como tablas Delta en el Lakehouse **Silver**.

**Entorno de ejecución:** Microsoft Fabric (notebook PySpark).

**Lakehouses necesarios:**
- Lakehouse predeterminado: `LH_Silver` (destino de las tablas).
- Lakehouse adicional adjunto: `LH_DP600_Lab` (origen de las tablas Bronze).

> Antes de ejecutar, configura las rutas ABFS de las tablas de origen en la sección siguiente. No publiques identificadores reales de tu workspace en el repositorio.


## 1. Configuración de las fuentes

Copia desde Fabric las rutas ABFS de las tablas `Bronze_Sales` y `Bronze_Customers` del Lakehouse `LH_DP600_Lab` y configúralas en las variables siguientes.

Las rutas se dejan como marcadores para evitar incluir identificadores específicos del entorno en el repositorio.


In [ ]:
# Sustituye estos marcadores por las rutas ABFS de tu propio entorno Fabric.
SALES_BRONZE_PATH = "<RUTA_ABFS_DE_BRONZE_SALES>"
CUSTOMERS_BRONZE_PATH = "<RUTA_ABFS_DE_BRONZE_CUSTOMERS>"

for nombre, ruta in {
    "SALES_BRONZE_PATH": SALES_BRONZE_PATH,
    "CUSTOMERS_BRONZE_PATH": CUSTOMERS_BRONZE_PATH,
}.items():
    if not ruta or ruta.startswith("<"):
        raise ValueError(f"Configura la ruta ABFS de {nombre} antes de ejecutar el notebook.")


## 2. Lectura de las tablas Bronze

Leemos las tablas Delta de origen mediante sus rutas ABFS.


In [ ]:
sales_bronze_df = (
    spark.read
    .format("delta")
    .load(SALES_BRONZE_PATH)
)

customers_bronze_df = (
    spark.read
    .format("delta")
    .load(CUSTOMERS_BRONZE_PATH)
)

print(f"Registros de ventas Bronze: {sales_bronze_df.count()}")
print(f"Registros de clientes Bronze: {customers_bronze_df.count()}")


## 3. Limpieza y validación de ventas

Aplicamos las siguientes reglas: tipos de datos explícitos, eliminación de duplicados por `SaleId`, descarte de registros sin claves o fecha válida, cantidades mayores que cero y precios unitarios no negativos. Por último, calculamos `TotalAmount`.


In [ ]:
from pyspark.sql.functions import col, round, to_date

sales_silver_df = (
    sales_bronze_df
    .withColumn("SaleId", col("SaleId").cast("integer"))
    .withColumn("SaleDate", to_date(col("SaleDate")))
    .withColumn("CustomerId", col("CustomerId").cast("integer"))
    .withColumn("Quantity", col("Quantity").cast("integer"))
    .withColumn("UnitPrice", col("UnitPrice").cast("double"))
    .dropDuplicates(["SaleId"])
    .filter(
        col("SaleId").isNotNull() &
        col("CustomerId").isNotNull() &
        col("SaleDate").isNotNull() &
        (col("Quantity") > 0) &
        (col("UnitPrice") >= 0)
    )
    .withColumn(
        "TotalAmount",
        round(col("Quantity") * col("UnitPrice"), 2)
    )
)


## 4. Limpieza y validación de clientes

Convertimos `CustomerId` a entero, eliminamos duplicados por esa clave y descartamos registros sin identificador o nombre de cliente.


In [ ]:
customers_silver_df = (
    customers_bronze_df
    .withColumn("CustomerId", col("CustomerId").cast("integer"))
    .dropDuplicates(["CustomerId"])
    .filter(
        col("CustomerId").isNotNull() &
        col("CustomerName").isNotNull()
    )
)


## 5. Persistencia en Silver

Guardamos los DataFrames como tablas Delta en el Lakehouse predeterminado (`LH_Silver`).

> **Importante:** `mode("overwrite")` sustituye el contenido existente de las tablas de destino. Se utiliza aquí para que el laboratorio sea reproducible; en un proceso productivo habría que definir una estrategia de carga incremental o actualización adecuada.


In [ ]:
(
    sales_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("Sales_Bronze")
)

(
    customers_silver_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable("Customers_Bronze")
)


## 6. Comprobaciones finales

Comprobamos los recuentos de las tablas generadas y mostramos una muestra de los resultados.


In [ ]:
sales_bronze_count = spark.table("Sales_Bronze").count()
customers_bronze_count = spark.table("Customers_Bronze").count()

print(f"Registros de ventas Bronze: {sales_bronze_count}")
print(f"Registros de clientes Bronze: {customers_bronze_count}")

display(spark.table("Sales_Bronze"))
display(spark.table("Customers_Bronze"))

## 7. Resultado esperado del laboratorio

Con los ficheros de ejemplo utilizados en este proyecto, el resultado esperado es de **10 registros de ventas** y **6 registros de clientes** en Silver. Estos recuentos sirven como comprobación del conjunto de datos de muestra; no son una garantía para otros datos de entrada.

**Tablas generadas:**
- `Sales_Bronze`
- `Customers_Bronze`
